In [1]:
# Load the Pickle file
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

In [9]:
# Load the trained model, Scaler, and OneHotEncoder from the Pickle file
model = load_model('churn_model.h5')

# Load the scaler and encoder from the Pickle file
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# label encoder for categorical features
with open('label_encoder_gender.pkl', 'rb') as f:
    encoder_gender = pickle.load(f)


# Onehot encode the categorical features using the loaded encoder
with open('onehot_encoder_geography.pkl', 'rb') as f:
    encoder_geography = pickle.load(f)                                                                                  



In [22]:
# Example Input DataFrame for prediction
input_data = pd.DataFrame({
    
    'CreditScore': [600],
    'Geography': ['France'],
    'Gender': ['Male'],
    'Age': [40],
    'Tenure': [3],
    'Balance': [60000],
    'NumOfProducts': [2],
    'HasCrCard': [1],
    'IsActiveMember': [1],
    'EstimatedSalary': [50000]
})

In [ ]:
# Apply label encoder to Gender
# If the encoder was a LabelEncoder fitted on the training 'Gender' values:
input_data['Gender'] = encoder_gender.transform(input_data['Gender'])

# Apply OneHotEncoder to Geography and concat (keep same feature names)
geo_arr = encoder_geography.transform(input_data[['Geography']]).toarray()
geo_cols = encoder_geography.get_feature_names_out(['Geography'])
geo_df = pd.DataFrame(geo_arr, columns=geo_cols, index=input_data.index)

# Combine and drop the original Geography column
input_data = pd.concat([input_data.drop(columns=['Geography']), geo_df], axis=1)

In [13]:
geo_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [27]:
input_data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [29]:
# Scale the input data using the loaded scaler
input_data = scaler.transform(input_data)

In [30]:
input_data

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [31]:
# Predict the churn probability using the trained model
churn_probability = model.predict(input_data)   
churn_probability

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


array([[0.03420914]], dtype=float32)

In [32]:
prediction_percentage = churn_probability[0][0] * 100
if prediction_percentage >= 50:
    print(f"The customer is likely to churn with a probability of {prediction_percentage:.2f}%.")
else:
    print(f"The customer is unlikely to churn with a probability of {prediction_percentage:.2f}%.")

The customer is unlikely to churn with a probability of 3.42%.
